# Offline inference

This Kaggle notebook predicts from precomputed test feature shards, averages the saved fold heads, and writes `/kaggle/working/submission.csv`. Attach all assets as Kaggle datasets and keep internet disabled.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PACKAGE = INPUT / 'rsna-knee-code' / 'rsna-knee-abnormality-detection' / 'src'
FEATURE_DIR = INPUT / 'rsna-knee-test-features'
MODELS = INPUT / 'rsna-knee-heads'
SAMPLE = INPUT / 'rsna-knee-abnormality-detection' / 'sample_submission.csv'
FINGERPRINT = 'REPLACE_WITH_FEATURE_FINGERPRINT'
PREDICTIONS = WORK / 'fold-ensemble-predictions.csv'
SUBMISSION = WORK / 'submission.csv'

assert PACKAGE.exists(), PACKAGE
assert MODELS.joinpath('metadata.json').is_file(), MODELS
assert SAMPLE.is_file(), SAMPLE
assert FINGERPRINT != 'REPLACE_WITH_FEATURE_FINGERPRINT'
FEATURE_SHARDS = sorted(FEATURE_DIR.glob('*.npz'))
assert FEATURE_SHARDS, FEATURE_DIR
os.environ['PYTHONPATH'] = str(PACKAGE)
PYTHON = sys.executable

In [ ]:
command = [
    PYTHON, '-m', 'rsna_knee', 'predict-fold-heads',
    '--models', str(MODELS),
    '--fingerprint', FINGERPRINT,
    '--output', str(PREDICTIONS),
    '--device', 'cuda',
]
for shard in FEATURE_SHARDS:
    command.extend(['--feature-shard', str(shard)])
subprocess.run(command, check=True)

In [ ]:
subprocess.run([
    PYTHON, '-m', 'rsna_knee', 'ensemble-submission',
    '--fold', str(PREDICTIONS),
    '--sample', str(SAMPLE),
    '--output', str(SUBMISSION),
], check=True)
assert SUBMISSION.is_file(), SUBMISSION
print('Submission ready at', SUBMISSION)